# Employee Attrition Analysis - Exploratory Data Analysis (EDA)

## Project Objective
The goal of this project is to analyze employee attrition patterns and build a predictive model to identify employees who are likely to leave the company. This notebook covers the initial **Exploratory Data Analysis (EDA)** phase: loading data, data cleaning, sanity checks, univariate/bivariate visualizations, feature correlation analysis, and feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Load dataset (handling potential BOM header)
dataset_path = '../data/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(dataset_path, encoding='utf-8-sig')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Data Loading Analysis
The dataset contains **1,470 rows** and **35 columns**. We successfully loaded the data with `encoding='utf-8-sig'` to handle the UTF-8 Byte Order Mark (BOM) in the first column name (`Age`), ensuring we have a clean `Age` column. The preview shows a mix of numerical features (like `Age`, `DailyRate`, `DistanceFromHome`) and categorical features (like `Attrition`, `BusinessTravel`, `Department`, `JobRole`).

In [ ]:
# Data Quality Check: Missing Values, Duplicates, and Types
print("Missing Values Count:", df.isnull().sum().sum())
print("Duplicate Rows Count:", df.duplicated().sum())
print("\nData Types:")
print(df.dtypes.value_counts())

### Data Quality Analysis
- **Missing Values**: There are **0** missing values in the dataset. This is highly clean data, requiring no immediate imputation at this stage.
- **Duplicates**: There are **0** duplicate rows, ensuring all observations are unique.
- **Data Types**: The features consist of **26 integer** columns and **9 object (categorical)** columns. We will need to encode the categorical features before modeling.

In [ ]:
# Class Distribution of Target Variable
attrition_counts = df['Attrition'].value_counts()
attrition_pct = df['Attrition'].value_counts(normalize=True) * 100
for label in attrition_counts.index:
    print(f"Attrition '{label}': {attrition_counts[label]} ({attrition_pct[label]:.2f}%)")

### Target Variable Class Distribution Analysis
The target column `Attrition` has **1,233 'No' (83.88%)** and **237 'Yes' (16.12%)**. This demonstrates a significant class imbalance (roughly 5:1 ratio). When training our classifier, we must consider this imbalance by choosing appropriate evaluation metrics (Precision, Recall, F1, ROC AUC) rather than relying solely on accuracy. We can also configure our Random Forest to balance class weights using `class_weight='balanced'` or `class_weight='balanced_subsample'`.

In [ ]:
# Identify Constant & Non-informative Columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print("Constant Columns (nunique <= 1):", constant_cols)
print(f"EmployeeNumber cardinality: {df['EmployeeNumber'].nunique()} / {len(df)} rows")

### Constant and Non-informative Columns Analysis
We identified the following columns as constant or unique identifiers:
- `EmployeeCount`: always 1
- `Over18`: always 'Y'
- `StandardHours`: always 80
- `EmployeeNumber`: unique identifier (1,470 unique values for 1,470 rows)

These columns contain no predictive signal and will be dropped during feature engineering to simplify the model and prevent overfitting.

In [ ]:
# Univariate Analysis: Key Numeric Fields
fig_age = px.histogram(df, x='Age', title='Employee Age Distribution',
                       color_discrete_sequence=['#2563eb'], nbins=30)
fig_age.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_age.show()

fig_income = px.histogram(df, x='MonthlyIncome', title='Employee Monthly Income Distribution',
                          color_discrete_sequence=['#16a34a'], nbins=35)
fig_income.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_income.show()

### Univariate Analysis
- **Age**: The distribution is approximately bell-shaped and centered around **35-38 years old**, representing a mature workforce. There are very few employees under 20 or over 60.
- **Monthly Income**: Right-skewed distribution. The majority of employees earn **under $5,000 per month**. A small subset of senior roles earns up to $20,000 per month, forming a long right tail.

In [ ]:
# Bivariate Analysis: Attrition vs Key Numeric Fields
fig_box_age = px.box(df, x='Attrition', y='Age', color='Attrition',
                     title='Age Distribution by Attrition Status',
                     color_discrete_map={'Yes': '#dc2626', 'No': '#16a34a'})
fig_box_age.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_box_age.show()

fig_box_inc = px.box(df, x='Attrition', y='MonthlyIncome', color='Attrition',
                     title='Monthly Income by Attrition Status',
                     color_discrete_map={'Yes': '#dc2626', 'No': '#16a34a'})
fig_box_inc.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_box_inc.show()

### Bivariate Analysis (Numeric)
- **Attrition by Age**: Employees who leave (Attrition = Yes) have a lower median age (**~32 years**) compared to employees who stay (Attrition = No, median **~36 years**). This indicates younger staff have higher attrition rates.
- **Attrition by Monthly Income**: Employees leaving have a significantly lower median income (approx. **$3,200**) compared to retained employees (approx. **$5,200**). Lower pay is strongly associated with attrition.

In [ ]:
# Bivariate Analysis: Attrition vs Key Categorical Fields
df_role = df.groupby(['JobRole', 'Attrition']).size().reset_index(name='Count')
fig_role = px.bar(df_role, x='JobRole', y='Count', color='Attrition',
                  title='Attrition Count by Job Role', barmode='group',
                  color_discrete_map={'Yes': '#dc2626', 'No': '#16a34a'})
fig_role.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_role.show()

df_ot = df.groupby(['OverTime', 'Attrition']).size().reset_index(name='Count')
fig_ot = px.bar(df_ot, x='OverTime', y='Count', color='Attrition',
                title='Attrition Count by Overtime Status', barmode='group',
                color_discrete_map={'Yes': '#dc2626', 'No': '#16a34a'})
fig_ot.update_layout(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_ot.show()

### Bivariate Analysis (Categorical)
- **Job Role**: Roles like **Sales Representative, Laboratory Technician, and Research Scientist** exhibit higher attrition proportions relative to their overall count. Managers and Research Directors have extremely low attrition.
- **Overtime**: Employees who work **OverTime = Yes** show a vastly higher attrition proportion. Over 30% of employees working overtime leave the company, compared to around 10% of those who do not.

In [ ]:
# Correlation Matrix
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove constant/ID columns
cols_to_exclude = ['EmployeeCount', 'StandardHours', 'EmployeeNumber']
num_cols = [c for c in num_cols if c not in cols_to_exclude]

corr_matrix = df[num_cols].corr()
fig_corr = px.imshow(corr_matrix, 
                     labels=dict(color="Correlation"),
                     x=num_cols,
                     y=num_cols,
                     color_continuous_scale='RdBu_r',
                     title='Correlation Matrix of Numerical Features')
fig_corr.update_layout(
    width=700,
    height=700,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(family='DM Sans, sans-serif', color='#71717a')
)
fig_corr.show()

### Correlation Analysis
Several numerical features show high positive correlation:
- `JobLevel` and `MonthlyIncome` are almost perfectly correlated (r = 0.95), which is expected since salary is tightly linked to job hierarchy.
- `YearsAtCompany`, `YearsInCurrentRole`, `YearsSinceLastPromotion`, and `YearsWithCurrManager` are strongly correlated (r >= 0.70).
- `TotalWorkingYears` is highly correlated with `JobLevel` (r = 0.78) and `Age` (r = 0.68).

We need to monitor multicollinearity, although Tree-based models (like Random Forests) are generally robust against it.

In [ ]:
# Feature Engineering & Cleaning
df_clean = df.copy()

# Drop constant and non-informative columns
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 1. Satisfaction Index (Average of 4 satisfaction dimensions)
df_clean['SatisfactionIndex'] = (df_clean['EnvironmentSatisfaction'] + 
                                 df_clean['JobSatisfaction'] + 
                                 df_clean['RelationshipSatisfaction'] + 
                                 df_clean['WorkLifeBalance']) / 4.0

# 2. Promotion Ratio (Years since last promotion / (Years at company + 1) to avoid division by zero)
df_clean['PromotionRatio'] = df_clean['YearsSinceLastPromotion'] / (df_clean['YearsAtCompany'] + 1)

# 3. Years per Company (Total working years / (Number of companies worked + 1) to avoid division by zero)
df_clean['YearsPerCompany'] = df_clean['TotalWorkingYears'] / (df_clean['NumCompaniesWorked'] + 1)

print(f"Original dataset shape: {df.shape}")
print(f"Clean dataset shape after feature engineering: {df_clean.shape}")
df_clean[['SatisfactionIndex', 'PromotionRatio', 'YearsPerCompany']].head()

### Feature Engineering Analysis
We dropped **4 non-informative columns** (`EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber`) and created **3 new features** to capture combined organizational dynamics:
1. `SatisfactionIndex`: The average of four satisfaction indicators, capturing the employee's overall contentment level.
2. `PromotionRatio`: The ratio of years since last promotion relative to their company tenure, showing whether an employee feels career stagnation.
3. `YearsPerCompany`: The average duration an employee spends per organization, highlighting job-hopping tendencies.

In [ ]:
# Save clean dataset
clean_data_dir = '../data'
os.makedirs(clean_data_dir, exist_ok=True)
clean_data_path = os.path.join(clean_data_dir, 'clean_data.csv')
df_clean.to_csv(clean_data_path, index=False)
print(f"Clean dataset saved to: {clean_data_path}")

### Final Summary

### Q&A
- **What are the key factors linked to attrition?** Based on visual analysis, younger employees, employees earning lower monthly income, those who work overtime, and those in specific roles (like Sales Representative and Laboratory Technician) show higher rates of attrition.
- **Are there columns that can be dropped safely?** Yes, `EmployeeCount`, `Over18`, `StandardHours`, and `EmployeeNumber` were identified as constant or non-informative and dropped.

### Data Analysis Key Findings
- **Class Imbalance**: The target variable has a significant class imbalance, with **83.88% No** and **16.12% Yes**.
- **Low Salary & Attrition**: Employees who leave have a median income of **~$3,200/month** compared to **~$5,200/month** for those who stay.
- **Age & Attrition**: The median age of employees leaving is **~32**, compared to **~36** for those staying.
- **Overtime & Attrition**: Working overtime is a strong indicator of departure, with **>30%** of overtime workers leaving the company.

### Insights or Next Steps
- In the next phase (Model Training), we must handle the class imbalance using strategies such as class weighting (`class_weight='balanced'`) during Random Forest training.
- Feature encoding must be established, dividing features into nominal types (e.g. `Department`, `JobRole` to be one-hot encoded) and ordinal types (e.g., satisfaction indicators, to be ordinally mapped).